# 원인 분해 (로직 트리 · 이슈 트리)

## 지표 지도 단계에서 좁혀진 문제를 원인별로 재분해

---

### 1. 좁혀진 문제

지표 지도 단계에서 계산한 지표를 다시 보면:

| 지표 | 값 |
|---|---:|
| 오퍼 완료율 (수신 대비) | 54.96% |
| 열람 후 완료율 (North Star Metric) | 50.53% |
| 완료 중 사전 열람이 없었던 비율 (Sure-thing) | 29.79% |

완료된 오퍼 중 약 30%는 고객이 오퍼를 열람하지도 않고 조건을 채운 것이다. 이는 오퍼가 없어도 어차피 발생했을 구매일 가능성이 높고, 이 경우 지급된 리워드는 실질적인 행동 변화 없이 소모된 비용이다.

> **좁혀진 질문: 완료 중 사전 열람이 없는 비율(Sure-thing)은 왜 발생하는가?**

이 질문을 로직 트리로 재분해하여 원인 후보를 나열하고, 정량 데이터로 검증 가능한 가지와 그렇지 않은 가지를 구분한다.

---

### 2. 이슈 트리 구조 (MECE)

문제를 4개 갈래로 분해한다. 각 갈래는 서로 다른 층위의 원인을 다루므로 겹치지 않고, 합쳐서 오퍼 캠페인의 주요 구성요소를 포괄한다.

```
완료의 30%는 왜 Sure-thing으로 새는가
├── A. 노출 채널 설계 (오퍼에 포함된 채널 조합)
│   ├── A1. 소셜 채널 부재 → 열람율 급감 → Sure-thing 증가
│   └── A2. 모바일 채널 부재 → 열람율 급감 → Sure-thing 증가
├── B. 오퍼 조건 설계 (난이도·유형)
│   ├── B1. 난이도(difficulty)가 높을수록 Sure-thing 증가
│   └── B2. 오퍼 유형(BOGO vs 할인) 자체가 영향을 준다
├── C. 고객 특성 (인구통계)
│   ├── C1. 소득이 높을수록 Sure-thing 증가
│   └── C2. 가입기간(로열티)이 길수록 Sure-thing 증가
└── D. 인지·타이밍 (정성적 요인 — 정량 데이터로 검증 불가)
    ├── D1. 고객이 오퍼 도착 전 이미 구매를 결심했다
    └── D2. 푸시 알림이 꺼져 있거나 스팸으로 인식돼 열지 않는다
```

신호가 강한 갈래(A)만 한 단계 더 파고, 나머지는 첫 단계 판정에서 멈춘다 — 모든 가지를 동일한 깊이로 파지 않는다.

---

## 3. 데이터 준비 (오퍼 수신 단위 재구성)

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('.')

portfolio = pd.read_json(DATA_DIR / 'portfolio.json', lines=True)
profile = pd.read_json(DATA_DIR / 'profile.json', lines=True)
transcript = pd.read_json(DATA_DIR / 'transcript.json', lines=True)

transcript_work = transcript.copy()
transcript_work['offer_id'] = transcript_work['value'].apply(
    lambda x: x.get('offer id', x.get('offer_id'))
)

# 오퍼 유효기간을 시간 단위로 변환
duration_hours = (portfolio.set_index('id')['duration'] * 24).to_dict()

# 결제를 제외한 오퍼 행동만 선택
offer_events = transcript_work.loc[
    transcript_work['event'] != 'transaction',
    ['person', 'event', 'time', 'offer_id']
].copy()

event_order = {'offer received': 0, 'offer viewed': 1, 'offer completed': 2}
offer_events['event_order'] = offer_events['event'].map(event_order)
offer_events = offer_events.sort_values(['person', 'offer_id', 'time', 'event_order'])

offer_instances = []
for (person, offer_id), group in offer_events.groupby(['person', 'offer_id'], sort=False):
    received_instances = []
    for row in group.itertuples(index=False):
        if row.event == 'offer received':
            received_instances.append({
                'customer_id': person, 'offer_id': offer_id,
                'received_time': row.time,
                'expiry_time': row.time + duration_hours[offer_id],
                'viewed_time': None, 'completed_time': None
            })
        elif row.event == 'offer viewed':
            candidates = [i for i in received_instances
                          if i['received_time'] <= row.time <= i['expiry_time'] and i['viewed_time'] is None]
            if candidates:
                max(candidates, key=lambda x: x['received_time'])['viewed_time'] = row.time
        elif row.event == 'offer completed':
            candidates = [i for i in received_instances
                          if i['received_time'] <= row.time <= i['expiry_time'] and i['completed_time'] is None]
            if candidates:
                max(candidates, key=lambda x: x['received_time'])['completed_time'] = row.time
    offer_instances.extend(received_instances)

offer_funnel = pd.DataFrame(offer_instances)

offer_funnel = offer_funnel.merge(
    portfolio[['id', 'offer_type', 'reward', 'difficulty', 'duration', 'channels']],
    left_on='offer_id', right_on='id', how='left'
).drop(columns='id')

offer_funnel['viewed'] = offer_funnel['viewed_time'].notna()
offer_funnel['completed'] = offer_funnel['completed_time'].notna()
offer_funnel['viewed_before_completed'] = (
    offer_funnel['viewed'] & offer_funnel['completed']
    & (offer_funnel['viewed_time'] <= offer_funnel['completed_time'])
)
offer_funnel['sure_thing'] = (
    offer_funnel['completed'] & ~offer_funnel['viewed_before_completed']
)

print('오퍼 수신 단위:', len(offer_funnel))


오퍼 수신 단위: 76277


In [2]:
# 원인분해에 필요한 고객 특성 결합 (인구통계 결측 고객 제외 — 앞선 지표 계산 단계의 주의사항 참고)
profile_clean = profile[profile['age'] != 118].copy()
profile_clean['income_seg'] = pd.cut(
    profile_clean['income'],
    [0, 40000, 60000, 80000, 100000, 1e9],
    labels=['<40k', '40-60k', '60-80k', '80-100k', '100k+']
)
profile_clean['member_year'] = profile_clean['became_member_on'].astype(str).str[:4].astype(int)

# 채널 조합 플래그
offer_funnel['n_channels'] = offer_funnel['channels'].apply(len)
offer_funnel['has_social'] = offer_funnel['channels'].apply(lambda c: 'social' in c)
offer_funnel['has_mobile'] = offer_funnel['channels'].apply(lambda c: 'mobile' in c)

df = offer_funnel.merge(
    profile_clean[['id', 'gender', 'income_seg', 'member_year']],
    left_on='customer_id', right_on='id', how='inner'
)

incentive = df[df['offer_type'].isin(['bogo', 'discount'])]
completed = incentive[incentive['completed']]

print(f'인구통계 결합 후 분석 대상(인센티브형, 결측 제외): {len(incentive):,}건')
print(f'그중 완료 건: {len(completed):,}건')


인구통계 결합 후 분석 대상(인센티브형, 결측 제외): 53,201건
그중 완료 건: 32,421건


---
## 4. 갈래 A — 노출 채널 설계

In [3]:
view_by_social = incentive.groupby('has_social')['viewed'].mean().mul(100).round(1)
sure_by_social = completed.groupby('has_social')['sure_thing'].mean().mul(100).round(1)
n_by_social = completed.groupby('has_social').size()

print('[소셜 채널 유무별 열람율]')
print(view_by_social)
print('\n[소셜 채널 유무별 Sure-thing 비율]')
print(sure_by_social)
print('\n[표본 크기]')
print(n_by_social)

print('\n---')
view_by_mobile = incentive.groupby('has_mobile')['viewed'].mean().mul(100).round(1)
print('[모바일 채널 유무별 열람율]')
print(view_by_mobile)
print('(모바일 미포함 오퍼는 포트폴리오 내 1종뿐 — 표본 일반화 주의)')


[소셜 채널 유무별 열람율]
has_social
False    45.3
True     94.0
Name: viewed, dtype: float64

[소셜 채널 유무별 Sure-thing 비율]
has_social
False    52.9
True     17.9
Name: sure_thing, dtype: float64

[표본 크기]
has_social
False    11475
True     20946
dtype: int64

---
[모바일 채널 유무별 열람율]
has_mobile
False    32.8
True     81.9
Name: viewed, dtype: float64
(모바일 미포함 오퍼는 포트폴리오 내 1종뿐 — 표본 일반화 주의)


**판정**

| 가지 | 신호 | 판정 | 근거 |
|---|---|---|---|
| A1. 소셜 채널 부재 → Sure-thing 증가 | 열람율 45.3%→94.0%, Sure-thing 52.9%→17.9% | **채택** | 소셜 미포함 오퍼가 3종에 걸쳐 있어 표본이 여러 오퍼에 분산됨. 효과크기가 모든 후보 중 가장 큼 |
| A2. 모바일 채널 부재 → Sure-thing 증가 | 열람율 32.8%→81.9% | **보류** | 해당 오퍼가 1종뿐이라 오퍼 개별 특성과 분리 불가 |

---
## 5. 갈래 B — 오퍼 조건 설계 (교란 주의)

In [4]:
sure_by_difficulty = completed.groupby('difficulty')['sure_thing'].mean().mul(100).round(1)
n_by_difficulty = completed.groupby('difficulty').size()
print('[난이도별 Sure-thing 비율]')
print(sure_by_difficulty)
print(n_by_difficulty)

# 난이도=20 오퍼가 채널 구성과 겹치는지 확인 (교란변수 체크)
print('\n[difficulty=20 오퍼의 채널 구성]')
print(portfolio.loc[portfolio['difficulty']==20, ['offer_type','difficulty','channels']])

print('\n[오퍼 유형별 Sure-thing 비율]')
print(completed.groupby('offer_type')['sure_thing'].mean().mul(100).round(1))


[난이도별 Sure-thing 비율]
difficulty
5     34.6
7     15.4
10    26.0
20    61.2
Name: sure_thing, dtype: float64
difficulty
5      8291
7      4883
10    15867
20     3380
dtype: int64

[difficulty=20 오퍼의 채널 구성]
  offer_type  difficulty      channels
4   discount          20  [web, email]

[오퍼 유형별 Sure-thing 비율]
offer_type
bogo        29.5
discount    31.0
Name: sure_thing, dtype: float64


**판정**

| 가지 | 신호 | 판정 | 근거 |
|---|---|---|---|
| B1. 난이도가 높을수록 Sure-thing 증가 | difficulty=20 오퍼 61.2% vs 나머지 15.4~34.6% | **보류** | difficulty=20 오퍼가 동시에 소셜·모바일이 모두 빠진 유일한 오퍼 — 갈래 A의 채널 효과와 뒤섞여 있어 독립 효과로 단정 불가. 채널을 통제한 재분석 필요 |
| B2. 오퍼 유형(BOGO vs 할인) 자체가 영향을 준다 | 차이 약 1.4%p | **기각** | 실질적으로 무의미한 차이 |

---
## 6. 갈래 C — 고객 특성

In [5]:
sure_by_income = completed.groupby('income_seg', observed=True)['sure_thing'].mean().mul(100).round(1)
n_by_income = completed.groupby('income_seg', observed=True).size()
print('[소득 구간별 Sure-thing 비율]')
print(sure_by_income)
print(n_by_income)

print('\n[가입연도별 Sure-thing 비율]')
print(completed.groupby('member_year')['sure_thing'].mean().mul(100).round(1))


[소득 구간별 Sure-thing 비율]
income_seg
<40k       31.0
40-60k     28.2
60-80k     28.7
80-100k    30.6
100k+      40.7
Name: sure_thing, dtype: float64
income_seg
<40k        3375
40-60k      8705
60-80k     10541
80-100k     6977
100k+       2823
dtype: int64

[가입연도별 Sure-thing 비율]
member_year
2013    24.6
2014    23.7
2015    30.5
2016    30.2
2017    31.0
2018    30.9
Name: sure_thing, dtype: float64


**판정**

| 가지 | 신호 | 판정 | 근거 |
|---|---|---|---|
| C1. 소득이 높을수록 Sure-thing 증가 | 100k+ 40.7% vs 나머지 28.2~31.0% | **채택** | 5개 소득 구간에 걸쳐 일관된 패턴, 표본 충분 |
| C2. 가입기간(로열티)이 길수록 Sure-thing 증가 | 2013년 가입 24.6% vs 2018년 가입 30.9% | **기각** | 가설과 반대 방향 — 오래된 회원이 오히려 더 낮음 |

---
## 7. 갈래 D — 인지·타이밍 (정량 데이터로 검증 불가)

이 갈래는 `offer_funnel`에 남아 있는 어떤 컬럼으로도 직접 검증할 수 없다. 로그에는 "고객이 언제 구매를 결심했는지", "알림이 실제로 기기에 도달했는지"가 기록되지 않기 때문이다.

| 가지 | 검증 불가 사유 | 필요한 정성 자료 |
|---|---|---|
| D1. 고객이 오퍼 도착 전 이미 구매를 결심했다 | 구매 의사결정 시점은 로그에 없음 | 구매 의사결정 시점 설문, CS 문의 로그 |
| D2. 푸시 알림이 꺼져 있거나 스팸으로 인식돼 열지 않는다 | 실제 알림 도달·확인 여부는 기록되지 않음 | 앱 알림 설정 로그, 실제 푸시 전달(delivery) 로그 |

이 갈래는 억지로 데이터를 끼워 맞추지 않고 검증 불가로 명시한다.

---
## 8. 종합 우선순위

| 우선순위 | 갈래 | 판정 | 다음 액션 |
|---|---|---|---|
| 1순위 | A1. 노출 채널 설계 (소셜 채널 부재) | 채택 | 채널 조합별 A/B 테스트 설계 — 소셜 채널 추가 시 Sure-thing 감소 여부 검증 |
| 2순위 | C1. 고객 특성 (고소득 세그먼트) | 채택 | 고소득 세그먼트는 오퍼 지급 없이도 유지될 가능성 — 리워드 축소 실험 검토 |
| 보류 | B1. 오퍼 난이도 | 보류(교란) | 채널을 통제한 재분석 후 재평가 |
| 보류 | A2. 모바일 채널 부재 | 보류(표본부족) | 유사 오퍼 추가 출시 후 재검증 |
| 정성조사 필요 | D. 인지·타이밍 | 검증불가 | CS 로그 확보, 고객 설문 설계 |
| 기각 | B2. 오퍼 유형, C2. 가입기간 | 기각 | 추가 조사 불필요 |

**결론**: 초기 데이터 신호로 볼 때, Sure-thing 낭비를 줄이기 위한 첫 액션은 **소셜 채널이 빠진 오퍼의 채널 구성을 보완하는 것**이다. 효과크기가 가장 크고, 여러 오퍼에 걸쳐 일관되게 나타나 표본 신뢰도도 가장 높다. 난이도 효과는 채널과 교란되어 있어 성급히 단정하지 않는다.